In [50]:
import pandas as pd
import sqlite3
import json


In [51]:
customers = pd.DataFrame({
 "CustomerID": [
 "C001", "C002", "C003", "C004",
 "C004", "C005", "C006", "C007",
 "C008", "C009", "C010"
 ],
 "CustomerName": [
 "Amina", "Brian", "Carol", "David",
 "David", "Esther", "Faith", "George",
 "Halima", "Ian", "Jane"
 ],
 "Segment": [
 "Retail", "Corporate", "Retail", "SME",
 "SME", "Corporate", "SME", "Retail",
 "Corporate", "Retail", "SME"
 ],
 "RegionCode": [
 "NRB", "MSA", "NRB", "KSM",
 "KSM", "NKR", "NRB", "MSA",
 "KSM", "NKR", "NRB"
 ]
})
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [52]:
customers.to_csv(
 "customers.csv",
 index=False
)


In [53]:
orders = pd.DataFrame({
 "OrderID": [
 "O001", "O002", "O003", "O004", "O005",
 "O006", "O007", "O008", "O009", "O010",
 "O011", "O012", "O013", "O014", "O015"
 ],
 "CustomerID": [
 "C001", "C002", "C001", "C003", "C004",
 "C005", "C006", "C002", "C007", "C008",
 "C009", "C010", "C006", "C003", "C999"
 ],
 "OrderAmount": [
      4500, 12500, 3200, 6800, 9100,
 15000, 7600, 8300, 2500, 11000,
 5400, 6200, 4300, 7200, 9900
 ],
 "ProductCategory": [
 "Electronics", "Office", "Home", "Electronics", "Office",
 "Electronics", "Home", "Office", "Home", "Electronics",
 "Office", "Home", "Electronics", "Office", "Electronics"
 ]
})
orders

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [54]:
orders.to_csv(
 "orders.csv",
 index=False
)


In [55]:
regions = [ 
{"RegionCode": "NRB", "RegionName": "Nairobi"},
 {"RegionCode": "MSA", "RegionName": "Mombasa"},
 {"RegionCode": "KSM", "RegionName": "Kisumu"},
 {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
 json.dump(regions, file, indent=4)
 


In [ ]:
customers = pd.read_csv("customers.csv")
customers.head()
#one row represents one customer of the company.


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [57]:

customers.shape #customers dataset has 11 rows and 4 columns.

(11, 4)

In [58]:
orders = pd.read_csv("orders.csv")
orders.head()
#one row represents one order placed by a customer of the company.


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


In [ ]:

orders.shape #orders dataset has 15 rows and 4 columns.


(15, 4)

The two datasets are not at the same aggregation grain. Each row in the customers dataset represents one customer, while each row in the orders dataset represents one order placed by a customer. Therefore, one customer can have multiple orders.

In [60]:
customers["CustomerID"].duplicated().sum()


np.int64(1)

In [ ]:
customers[
 customers["CustomerID"].duplicated(keep=False)
]


,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Duplicated customer IDs are dangerous because they multiply the number of orders for a customer when we join the two datasets, leading to wrong number of orders and wrong total order amount for a customer.

In [ ]:
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

Customer c999 appears in orders but not in customers.
C999 should not be deleted yet because it could mean that the system has an issue or that the customer is new and has not been added to the customers dataset yet.

In [63]:
conn = sqlite3.connect("retail_lab.db")

In [64]:
customers.to_sql(
 "customers",
 conn,
 if_exists="replace",
 index=False
)
orders.to_sql(
 "orders",
 conn,
 if_exists="replace",
 index=False
)


15

In [65]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


In [66]:
query = """
SELECT * FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [67]:
query = """
SELECT "OrderID","CustomerID","OrderAmount" FROM orders;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


In [68]:
query= """
SELECT "CustomerID", "CustomerName","Segment" FROM customers
WHERE "Segment"= "Retail";
"""
pd.read_sql_query(query, conn)


,CustomerID,CustomerName,Segment
0,C001,Amina,Retail
1,C003,Carol,Retail
2,C007,George,Retail
3,C009,Ian,Retail


In [69]:
query="""
SELECT * FROM orders
WHERE "OrderAmount" > 8000;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [70]:
query = """
SELECT * FROM orders
WHERE "ProductCategory" = "Electronics";

"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [71]:
query = """
SELECT * FROM orders
WHERE "ProductCategory" = "Electronics"
AND "OrderAmount" > 8000;

"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


In [72]:
query = """
SELECT
 SUM(OrderAmount) AS TotalSales
FROM orders;
"""
pd.read_sql_query(query, conn)

,TotalSales
0,113500


In [73]:
query = """
SELECT
 ProductCategory,
 SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


In [ ]:
query = """
SELECT
ProductCategory,
COUNT(OrderID) AS NumberofOrders,
SUM(OrderAmount) AS TotalSales,
AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""
pd.read_sql_query(query, conn)


,ProductCategory,NumberofOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


Electronics has the highest total sales and average order value of 51500 and 8583.33 respectively, while Home has the lowest total sales and average order value of 19500 and 4875.00 respectively.
Electronics has the highest number of orders of 6, Office has the second highest with 5, and Home has the lowest number of orders of 4.


In [79]:
query = """
SELECT
 o.OrderID,
 o.CustomerID,
 c.CustomerName,
 c.Segment,
 o.OrderAmount
FROM orders o
LEFT JOIN customers c
 ON o.CustomerID = c.CustomerID;
"""
pd.read_sql_query(query, conn)

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [80]:
query = """
SELECT
 o.OrderID,
 o.CustomerID,
 c.CustomerName,
 c.Segment,
 o.OrderAmount
FROM orders o
LEFT JOIN customers c
 ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined



,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [ ]:
len(orders)

15

In [ ]:
len(joined) 

16

The orders table has 15 rows, while the joined table has 16 rows because customerID 4 is duplicated in the customers datasethence it is also duplicated in the joined dataset. 
The unmatched customer record is c999. Its CustomerName and Segment are null because it does not exist in the customers dataset. The LEFT JOIN keeps the order because it comes from the orders table.

In [85]:
orders["OrderAmount"].sum()

np.int64(113500)

In [ ]:
joined["OrderAmount"].sum()

np.int64(122600)

The two sales totals are not equal.
The total order amount after the join is greater than before because C004's order was duplicated in the joined dataset due to the duplicated customerID in customers,
causing its order, O005, worth KSh 9,100, to be included twice after the JOIN.
Total reconciliation is an important validation step after a JOIN because it ensures that the total order amount remains consistent before and after the join operation.


In [100]:
customers_clean = (
    customers.drop_duplicates(subset="CustomerID")
)

customers_clean.to_sql(
 "customers_clean",
 conn,
 if_exists="replace",
 index=False
)


10

In [102]:
customers_clean

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR
10,C010,Jane,SME,NRB


In [ ]:

query = """
SELECT
 o.OrderID,
 o.CustomerID,
 c.CustomerName,
 c.Segment,
 o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
 ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query, conn)


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [115]:
clean_joined.to_sql("clean_joined", conn, if_exists="replace", index=False)

15

In [116]:
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [117]:
len(orders), len(clean_joined)

(15, 15)

In [ ]:
orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum()

(np.int64(113500), np.int64(113500))

The corrected join has preserved the total order value of 113500 and row count of 15, which is the same as before the join. 
This indicates that the duplicated customerID issue has been resolved and the join is now accurate.


In [120]:
query = """
SELECT Segment,
COUNT(OrderID) AS Orders,
SUM(OrderAmount) AS TotalSales,
AVG(OrderAmount) AS AverageOrderValue
FROM clean_joined
GROUP BY Segment;
"""
pd.read_sql_query(query, conn)

,Segment,Orders,TotalSales,AverageOrderValue
0,None,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


Corporate customers generated the highest total sales of KSh 46,800 from four orders. Retail customers made the highest number of orders, with six orders, but their average order value was lower at KSh 4,933.33. SME customers generated KSh 27,200 in sales from four orders, while one unmatched order worth KSh 9,900 could not be assigned to a customer segment.

In [121]:
regions_df = pd.read_json("regions.json")
regions_df


,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [ ]:
customers_regions = customers_clean.merge(
 regions_df,
 on="RegionCode",
 how="left"
)
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


Each customer now has a readable RegionName corresponding to their RegionCode.
If a RegionCode did not exist in regions.json, the left merge would keep the customer record but produce a NULL value for RegionName because there would be no matching region in the other table.


In [124]:
customers_regions.to_sql("customers_regions", conn, if_exists="replace", index=False)

10

In [129]:
query = """
SELECT
 cj.OrderID,
 cj.CustomerID,
 cj.CustomerName,
 cr.RegionCode,
 cr.RegionName,
 cj.OrderAmount
FROM clean_joined cj

LEFT JOIN customers_regions cr
 ON cj.CustomerID = cr.CustomerID;
"""
fmerge= pd.read_sql_query(query, conn)
fmerge



,OrderID,CustomerID,CustomerName,RegionCode,RegionName,OrderAmount
0,O001,C001,Amina,NRB,Nairobi,4500
1,O002,C002,Brian,MSA,Mombasa,12500
2,O003,C001,Amina,NRB,Nairobi,3200
3,O004,C003,Carol,NRB,Nairobi,6800
4,O005,C004,David,KSM,Kisumu,9100
5,O006,C005,Esther,NKR,Nakuru,15000
6,O007,C006,Faith,NRB,Nairobi,7600
7,O008,C002,Brian,MSA,Mombasa,8300
8,O009,C007,George,MSA,Mombasa,2500
9,O010,C008,Halima,KSM,Kisumu,11000


In [132]:
fmerge.to_sql("fmerge", conn, if_exists="replace", index=False)

15

In [ ]:
query = """
SELECT
RegionName,
COUNT(OrderID) AS NumberOfOrders,
SUM(OrderAmount) AS TotalSales
FROM fmerge
GROUP BY RegionName;
"""
region_sales = pd.read_sql_query(query, conn)
region_sales


,RegionName,NumberOfOrders,TotalSales
0,None,1,9900
1,Kisumu,2,20100
2,Mombasa,3,23300
3,Nairobi,7,39800
4,Nakuru,2,20400


Nairobi generated the highest total sales of 39800.
No, Management should not automatically conclude that Nairobi is the “best-performing” region because the dataset is small, with only 15 orders, 
so the results may not represent the company's overall regional performance.
There is also no time information, such as order dates, so management cannot determine whether Nairobi's high sales are consistent over time
or were caused by a short-term increase.


The biggest data-quality problem was the duplicate CustomerID C004 in the customer dataset. This caused the C004 order to be duplicated during the JOIN, resulting in an incorrect row count and overstated sales totals. 
Totals should be checked after joining tables to ensure that records have not been duplicated or lost.
The Corporate segment generated the most sales, with total sales of 46800. 
One limitation is that the analysis uses a small dataset, so the findings may not represent the company's overall sales performance.